### Метрика

In [ ]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_true, y_pred, average="macro")

In [ ]:
import os
import numpy as np
import pandas as pd
import polars as pl

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier, Pool
import mlflow
import mlflow.catboost

In [ ]:
DATA_DIR = "data"

TRAIN_MAIN_PATH = f"{DATA_DIR}/train_main.parquet"
TEST_MAIN_PATH  = f"{DATA_DIR}/test_main.parquet"
TRAIN_EXTRA_PATH = f"{DATA_DIR}/train_extra.parquet"
TEST_EXTRA_PATH  = f"{DATA_DIR}/test_extra.parquet"
TARGET_PATH      = f"{DATA_DIR}/train_target.parquet"

SPLIT_PATH = "split.parquet"

RANDOM_STATE = 1234

In [ ]:
train_main = pl.read_parquet(TRAIN_MAIN_PATH)
test_main  = pl.read_parquet(TEST_MAIN_PATH)
targets    = pl.read_parquet(TARGET_PATH)

target_cols = [c for c in targets.columns if c != "customer_id"]
main_cols = [c for c in train_main.columns if c != "customer_id"]

print(train_main.shape, test_main.shape, targets.shape)
print("n_targets =", len(target_cols))
print("n_main_features =", len(main_cols))

In [ ]:
customer_ids = train_main["customer_id"].to_list()

train_ids, val_ids = train_test_split(
    customer_ids,
    test_size=0.2,
    random_state=RANDOM_STATE
)

split_df = pl.DataFrame({
    "customer_id": customer_ids,
    "split": ["train" if cid in set(train_ids) else "val" for cid in customer_ids]
})

split_df.write_parquet(SPLIT_PATH)
print("saved:", SPLIT_PATH)

In [ ]:
split_df = pl.read_parquet(SPLIT_PATH)

train_ids = split_df.filter(pl.col("split") == "train")["customer_id"]
val_ids   = split_df.filter(pl.col("split") == "val")["customer_id"]

In [ ]:
def multilabel_macro_auc(y_true_df: pd.DataFrame, y_pred: np.ndarray, target_cols: list[str]) -> float:
    """
    y_true_df: DataFrame с таргетами, shape (n_samples, n_targets)
    y_pred: numpy array, shape (n_samples, n_targets)
    target_cols: порядок таргетов
    """
    aucs = []
    for i, col in enumerate(target_cols):
        if y_true_df[col].nunique() == 2:
            aucs.append(roc_auc_score(y_true_df[col], y_pred[:, i]))
    return float(np.mean(aucs))

In [ ]:
def make_train_val(train_df: pl.DataFrame):
    train_with_split = train_df.join(split_df, on="customer_id", how="left")

    df_train = train_with_split.filter(pl.col("split") == "train").drop("split")
    df_val   = train_with_split.filter(pl.col("split") == "val").drop("split")

    return df_train, df_val

In [ ]:
def get_extra_columns():
    schema = pl.scan_parquet(TRAIN_EXTRA_PATH).collect_schema().names()
    return [c for c in schema if c != "customer_id"]

EXTRA_ALL_COLS = get_extra_columns()
print("n_extra =", len(EXTRA_ALL_COLS))

In [ ]:
def load_extra(selected_cols: list[str]):
    cols = ["customer_id"] + selected_cols

    train_extra = pl.scan_parquet(TRAIN_EXTRA_PATH).select(cols).collect()
    test_extra  = pl.scan_parquet(TEST_EXTRA_PATH).select(cols).collect()

    return train_extra, test_extra

In [ ]:
def build_dataset(
    use_extra: bool = False,
    extra_cols: list[str] | None = None
):
    train_df = train_main.clone()
    test_df  = test_main.clone()

    if use_extra:
        if extra_cols is None:
            extra_cols = EXTRA_ALL_COLS

        train_extra, test_extra = load_extra(extra_cols)
        train_df = train_df.join(train_extra, on="customer_id", how="left")
        test_df  = test_df.join(test_extra, on="customer_id", how="left")

    train_df = train_df.join(targets, on="customer_id", how="left")
    return train_df, test_df

In [ ]:
def prepare_for_catboost(df: pl.DataFrame, feature_cols: list[str]):
    X = df.select(feature_cols).to_pandas()

    cat_cols = [c for c in feature_cols if c.startswith("cat_feature")]
    for c in cat_cols:
        X[c] = X[c].astype("string").fillna("missing")

    return X, cat_cols

In [ ]:
def make_model():
    return CatBoostClassifier(
        iterations=10,
        depth=4,
        learning_rate=0.25,
        loss_function='MultiLogloss',
        nan_mode='Min',
        random_seed=1234,
        task_type='GPU',
        devices='0',
        verbose=1
    )

In [ ]:
def train_and_eval(train_df: pl.DataFrame, feature_cols: list[str], exp_name: str):
    df_train, df_val = make_train_val(train_df)

    X_train_pd, cat_cols = prepare_for_catboost(df_train, feature_cols)
    X_val_pd, _          = prepare_for_catboost(df_val, feature_cols)

    y_train_pd = df_train.select(target_cols).to_pandas()
    y_val_pd   = df_val.select(target_cols).to_pandas()

    model = make_model()

    with mlflow.start_run(run_name=exp_name):
        mlflow.log_param("exp_name", exp_name)
        mlflow.log_param("n_features", len(feature_cols))
        mlflow.log_param("n_cat_features", len(cat_cols))

        train_pool = Pool(X_train_pd, label=y_train_pd, cat_features=cat_cols)
        val_pool   = Pool(X_val_pd, label=y_val_pd, cat_features=cat_cols)

        model.fit(train_pool)

        val_pred = model.predict_proba(val_pool)
        val_pred = np.asarray(val_pred)

        macro_auc = multilabel_macro_auc(y_val_pd, val_pred, target_cols)
        mlflow.log_metric("val_macro_auc", macro_auc)

        print(f"{exp_name} | macro_auc = {macro_auc:.6f}")

        mlflow.catboost.log_model(model, artifact_path="model")

    return model, macro_auc, val_pred

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("catboost_feature_study")

train_df, test_df = build_dataset(use_extra=False)
feature_cols = [c for c in train_df.columns if c not in ["customer_id"] + target_cols]

model_main, score_main, pred_main = train_and_eval(
    train_df=train_df,
    feature_cols=feature_cols,
    exp_name="E0_main_only"
)

In [ ]:
train_df, test_df = build_dataset(use_extra=True, extra_cols=EXTRA_ALL_COLS[:300])
feature_cols = [c for c in train_df.columns if c not in ["customer_id"] + target_cols]

model_extra_small, score_extra_small, _ = train_and_eval(
    train_df=train_df,
    feature_cols=feature_cols,
    exp_name="E1_main_plus_extra300"
)

In [ ]:
def filter_constant_and_sparse(train_df: pl.DataFrame, feature_cols: list[str], missing_thr=0.98):
    sample = train_df.select(feature_cols).sample(
        n=min(50000, train_df.height),
        seed=RANDOM_STATE
    )

    keep_cols = []
    removed_constant = []
    removed_sparse = []

    n = sample.height

    for c in feature_cols:
        s = sample.get_column(c)

        if s.n_unique() <= 1:
            removed_constant.append(c)
            continue

        miss_rate = s.null_count() / n
        if miss_rate >= missing_thr:
            removed_sparse.append(c)
            continue

        keep_cols.append(c)

    return keep_cols, removed_constant, removed_sparse

In [ ]:
train_df, test_df = build_dataset(use_extra=True, extra_cols=EXTRA_ALL_COLS[:300])
feature_cols = [c for c in train_df.columns if c not in ["customer_id"] + target_cols]

clean_cols, removed_const, removed_sparse = filter_constant_and_sparse(
    train_df, feature_cols, missing_thr=0.98
)

print("removed constant:", len(removed_const))
print("removed sparse:", len(removed_sparse))
print("keep:", len(clean_cols))

model_clean, score_clean, _ = train_and_eval(
    train_df=train_df,
    feature_cols=clean_cols,
    exp_name="E2_clean_const_sparse"
)

In [ ]:
def filter_correlated_numeric(train_df: pl.DataFrame, feature_cols: list[str], corr_thr=0.98):
    num_cols = [c for c in feature_cols if c.startswith("num_feature")]

    if len(num_cols) < 2:
        return feature_cols, []

    sample = train_df.select(num_cols).sample(
        n=min(30000, train_df.height),
        seed=RANDOM_STATE
    ).to_pandas()

    corr = sample.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    to_drop = [c for c in upper.columns if (upper[c] > corr_thr).any()]
    keep_cols = [c for c in feature_cols if c not in set(to_drop)]

    return keep_cols, to_drop

In [ ]:
train_df, test_df = build_dataset(use_extra=True, extra_cols=EXTRA_ALL_COLS[:300])
feature_cols = [c for c in train_df.columns if c not in ["customer_id"] + target_cols]

clean_cols, removed_const, removed_sparse = filter_constant_and_sparse(train_df, feature_cols)
corr_cols, removed_corr = filter_correlated_numeric(train_df, clean_cols, corr_thr=0.98)

print("removed corr:", len(removed_corr))
print("keep after corr:", len(corr_cols))

model_corr, score_corr, _ = train_and_eval(
    train_df=train_df,
    feature_cols=corr_cols,
    exp_name="E3_clean_corr"
)

In [ ]:
def get_topk_by_importance(train_df: pl.DataFrame, feature_cols: list[str], top_k=300):
    df_train, _ = make_train_val(train_df)

    X_train_pd, cat_cols = prepare_for_catboost(df_train, feature_cols)
    y_train_pd = df_train.select(target_cols).to_pandas()

    model = make_model()
    model.fit(Pool(X_train_pd, label=y_train_pd, cat_features=cat_cols))

    importances = model.get_feature_importance()
    imp_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": importances
    }).sort_values("importance", ascending=False)

    top_cols = imp_df.head(top_k)["feature"].tolist()
    return top_cols, imp_df